# 04 — Output Parsers & Structured Output

A model returns text. But a *program* usually needs **data**: a number, a category, a
list, an object with named fields. Squinting at `.content` and hoping it's valid JSON is
fragile. This notebook covers two levels of getting usable output:

1. **Output parsers** — lightweight post-processing of the model's text (e.g. "just give
   me the string").
2. **Structured output** — the reliable approach: ask the model to return data matching a
   **Pydantic** schema, and get back a validated Python object.

If you know Python dataclasses, Pydantic will feel familiar — it's classes with typed,
validated fields.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("Model ready.")

## 1. `StrOutputParser` — get the string out

Chains often end by extracting `.content` from the `AIMessage`. `StrOutputParser` does
exactly that as a Runnable, so `prompt | model | parser` yields a **plain string** — much
nicer to work with than reaching into `.content` every time.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Give me a tagline for a {product}."),
])

chain = prompt | model | StrOutputParser()

result = chain.invoke({"product": "note-taking app"})
print(type(result))   # <class 'str'> — not an AIMessage
print(result)

That's the small win. The big win is making the model return *structured* data.

## 2. Structured output with Pydantic — the reliable way

`with_structured_output(Schema)` wraps the model so that, instead of free text, it returns
an instance of your schema. Under the hood LangChain tells Gemini exactly which fields and
types to produce and validates the result. You get a real Python object with typed
attributes.

Define the shape you want as a Pydantic model. **The `description` on each field is part
of the prompt** — it tells the model what to put there, so write it well.

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    '''Information about a person extracted from text.'''
    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years")
    occupation: str = Field(description="Their job or profession")

# Wrap the model so it must return a Person.
structured_model = model.with_structured_output(Person)

result = structured_model.invoke(
    "Maria Garcia is a 34-year-old architect living in Barcelona."
)
print(type(result))
print(result)
print("\nAccess fields like any object:")
print("  name:", result.name)
print("  age :", result.age, type(result.age))   # a real int

Notice `result.age` is an `int`, not the string `"34"`. The schema drove both the
extraction and the type conversion. No JSON parsing, no regex, no guesswork.

## 3. Richer schemas: optionals, lists, and nesting

Pydantic schemas can be as expressive as your data. Use:

- `Optional[...]` (or `| None`) with a default for fields that might be missing,
- `list[...]` for repeated values,
- nested models for structure within structure.

Here we extract a recipe.

In [ ]:
from typing import Optional

class Ingredient(BaseModel):
    name: str = Field(description="Ingredient name")
    quantity: str = Field(description="Amount, e.g. '2 cups' or '1 tbsp'")

class Recipe(BaseModel):
    '''A cooking recipe.'''
    title: str = Field(description="Name of the dish")
    servings: Optional[int] = Field(default=None, description="Number of servings, if stated")
    ingredients: list[Ingredient] = Field(description="All ingredients with quantities")
    steps: list[str] = Field(description="Ordered preparation steps")

recipe_model = model.with_structured_output(Recipe)

text = '''Classic Guacamole (serves 4): mash 3 ripe avocados with 1 lime's juice,
half a diced onion, a handful of chopped cilantro, and salt to taste. Combine everything
and serve immediately.'''

recipe = recipe_model.invoke(text)
print(recipe.title, "| servings:", recipe.servings)
print("\nIngredients:")
for ing in recipe.ingredients:
    print(f"  - {ing.quantity} {ing.name}")
print("\nSteps:")
for i, step in enumerate(recipe.steps, 1):
    print(f"  {i}. {step}")

## 4. Classification with `Literal` (enums)

For classification, constrain a field to a fixed set of values with `Literal`. The model
can only return one of them — no more parsing free-text labels or handling typos like
"Positive!" vs "positive".

In [ ]:
from typing import Literal

class Sentiment(BaseModel):
    '''Sentiment analysis of a customer review.'''
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Overall sentiment of the review"
    )
    confidence: float = Field(description="Confidence from 0.0 to 1.0")
    reason: str = Field(description="A short justification")

clf = model.with_structured_output(Sentiment)

reviews = [
    "The battery lasts forever and the screen is gorgeous!",
    "It works, I guess. Nothing special.",
    "Stopped charging after two weeks. Very disappointed.",
]
for r in reviews:
    s = clf.invoke(r)
    print(f"[{s.sentiment:8}] conf={s.confidence:.2f}  {r[:40]}...")

This is the production-grade version of the few-shot classifier from notebook 03: the
output type is *guaranteed*, so the surrounding code can trust it.

## 5. A practical extraction pipeline

Combine a prompt template with structured output to build a reusable extractor — for
example, pulling action items out of meeting notes. This is the kind of component you'd
drop into a real app.

In [ ]:
from typing import Optional

class ActionItem(BaseModel):
    task: str = Field(description="What needs to be done")
    owner: Optional[str] = Field(default=None, description="Person responsible, if named")
    due: Optional[str] = Field(default=None, description="Deadline, if mentioned")

class MeetingSummary(BaseModel):
    '''Structured summary of meeting notes.'''
    summary: str = Field(description="One-sentence summary of the meeting")
    action_items: list[ActionItem] = Field(description="Concrete follow-up tasks")

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "You extract structured summaries from meeting notes."),
    ("human", "{notes}"),
])

# A prompt piped into a structured model is itself a chain.
extractor = extract_prompt | model.with_structured_output(MeetingSummary)

notes = '''Standup, Monday. Priya will ship the login fix by Wednesday. We agreed to
postpone the redesign. Tom needs to email the vendor about pricing. Open question: do we
support dark mode?'''

result = extractor.invoke({"notes": notes})
print("Summary:", result.summary, "\n")
for item in result.action_items:
    print(f"- {item.task} (owner: {item.owner}, due: {item.due})")

## 6. When you don't want a class: `JsonOutputParser`

If you just want a dict rather than a typed object, `JsonOutputParser` parses the model's
text into Python data. You typically tell the model to emit JSON via the parser's format
instructions. `with_structured_output` is more robust (it constrains the model directly),
so prefer it — but `JsonOutputParser` is good to know, especially for streaming partial
JSON.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return ONLY valid JSON. {format_instructions}"),
    ("human", "List 3 popular Python web frameworks with a one-line description each, "
              "as a JSON array of objects with keys 'name' and 'description'."),
]).partial(format_instructions=parser.get_format_instructions())

chain = json_prompt | model | parser
data = chain.invoke({})
print(type(data))            # list / dict — native Python
for item in data:
    print("-", item["name"], "—", item["description"])

> **Gotcha.** The quality of structured output depends heavily on your field
> `description`s — they *are* instructions to the model. If a field comes back wrong,
> sharpen its description before anything else. Also keep schemas focused; very large or
> deeply nested schemas are harder for the model to fill correctly.

## Your turn

Five extraction-and-structuring tasks straight out of everyday apps — an invoice parser, a
résumé reader, a review analyzer, a sales-lead extractor, and a quiz generator. Try each in a
blank cell before opening its solution.

**Exercise 1 — Invoice parser.** Define nested Pydantic models `LineItem` (description, amount)
and `Invoice` (vendor, date, items, total) and extract them from a free-text invoice. Confirm
`total` is parsed as a `float`, not a string.

*Sample run (illustrative):*

```text
Acme Tools | total: 209.5 <class 'float'>
  cordless drill: $89.0
  drill-bit set: $24.5
  shipping: $7.0
```

<details><summary>Show solution</summary>

```python
class LineItem(BaseModel):
    description: str = Field(description="What was bought")
    amount: float = Field(description="Line total in dollars")

class Invoice(BaseModel):
    '''A parsed invoice.'''
    vendor: str = Field(description="Who issued the invoice")
    date: str = Field(description="Invoice date as written")
    items: list[LineItem] = Field(description="Each line item")
    total: float = Field(description="Grand total in dollars")

inv = model.with_structured_output(Invoice)
text = ("Acme Tools — Invoice dated 2026-03-14. 2x cordless drill $89.00, "
        "1x drill-bit set $24.50, shipping $7.00. Total due: $209.50.")
result = inv.invoke(text)
print(result.vendor, "| total:", result.total, type(result.total))
for it in result.items:
    print(f"  {it.description}: ${it.amount}")
```

</details>

**Exercise 2 — Résumé reader.** Build a `Candidate` model with `name`, `years_experience: int`,
`skills: list[str]`, and an `Optional[str]` email, then extract it from a short bio.

*Sample run (illustrative):*

```text
name='Jordan Lee' years_experience=6 skills=['Python', 'Go', 'PostgreSQL', 'Docker', 'AWS'] email='jordan@example.com'
```

<details><summary>Show solution</summary>

```python
from typing import Optional

class Candidate(BaseModel):
    '''A parsed résumé summary.'''
    name: str = Field(description="Candidate's full name")
    years_experience: int = Field(description="Total years of professional experience")
    skills: list[str] = Field(description="Technical skills mentioned")
    email: Optional[str] = Field(default=None, description="Email address if present")

reader = model.with_structured_output(Candidate)
cv = ("Jordan Lee, 6 years building backend systems in Python and Go. "
      "Skilled in PostgreSQL, Docker, and AWS. Reach me at jordan@example.com.")
print(reader.invoke(cv))
```

`years_experience` arrives as an `int` and `email` as `None` when missing — no parsing needed.

</details>

**Exercise 3 — Review analyzer.** Build a `ReviewInsight` model with `rating: int` (1–5), a
`recommend: Literal["yes", "no", "maybe"]`, and a `top_complaint` string, then run it on a
mixed product review.

*Sample run (illustrative):*

```text
rating=3 recommend='maybe' top_complaint='overheats while gaming'
```

<details><summary>Show solution</summary>

```python
from typing import Literal

class ReviewInsight(BaseModel):
    '''Structured read of a product review.'''
    rating: int = Field(description="Star rating the reviewer implies, 1 to 5")
    recommend: Literal["yes", "no", "maybe"] = Field(description="Would they recommend it")
    top_complaint: str = Field(description="The single biggest issue, or 'none'")

m = model.with_structured_output(ReviewInsight)
review = "Great camera and battery, but it overheats while gaming and the price is steep."
print(m.invoke(review))
```

`Literal` guarantees `recommend` is one of three values — safe to branch on in code.

</details>

**Exercise 4 — Sales-lead extractor.** Combine a prompt template with structured output to
pull a `Lead` (name, optional company, `interest` Literal, optional budget) out of an inbound
message — a reusable `prompt | model.with_structured_output(...)` chain.

*Sample run (illustrative):*

```text
name='Dana' company='Volt Inc.' interest='demo' budget_usd=5000
```

<details><summary>Show solution</summary>

```python
from typing import Optional, Literal

class Lead(BaseModel):
    '''A sales lead parsed from an inbound message.'''
    name: str = Field(description="Contact's name")
    company: Optional[str] = Field(default=None, description="Their company, if mentioned")
    interest: Literal["demo", "pricing", "support", "other"] = Field(description="What they want")
    budget_usd: Optional[int] = Field(default=None, description="Stated budget in USD, if any")

lead_chain = ChatPromptTemplate.from_messages([
    ("system", "You extract sales-lead details from inbound messages."),
    ("human", "{message}"),
]) | model.with_structured_output(Lead)

msg = "Hi, I'm Dana from Volt Inc. We'd like a demo next week; budget is around $5000."
print(lead_chain.invoke({"message": msg}))
```

</details>

**Exercise 5 — Quiz generator.** Use `JsonOutputParser` to make the model return a JSON array
of multiple-choice questions (keys `question`, `options`, `answer`) about a topic, then print
them.

*Sample run (illustrative):*

```text
What does HTTP status code 404 mean?
   - OK
   - Not Found
   - Forbidden
   - Internal Server Error
   answer: Not Found

Which status code indicates a successful request?
   - 200
   - 301
   - 500
   - 403
   answer: 200

What does a 500 status code represent?
   - Bad Request
   - Unauthorized
   - Internal Server Error
   - Moved Permanently
   answer: Internal Server Error
```

<details><summary>Show solution</summary>

```python
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return ONLY valid JSON. {format_instructions}"),
    ("human", "Create 3 multiple-choice questions about {topic}. JSON array of objects "
              "with keys 'question', 'options' (list of 4 strings), and 'answer'."),
]).partial(format_instructions=parser.get_format_instructions())

quiz = (quiz_prompt | model | parser).invoke({"topic": "HTTP status codes"})
for q in quiz:
    print(q["question"])
    for opt in q["options"]:
        print("   -", opt)
    print("   answer:", q["answer"], "\n")
```

For a guaranteed schema you'd reach for `with_structured_output`; `JsonOutputParser` shines
when you want plain dicts or to stream partial JSON.

</details>

## Summary

- **`StrOutputParser`** ends a chain with a clean `str` instead of an `AIMessage`.
- **`with_structured_output(PydanticModel)`** is the reliable way to get data: the model
  returns a validated object with typed fields.
- Use `Optional`, `list[...]`, nested models, and `Literal` to describe exactly the shape
  and allowed values you need.
- Field **`description`s are prompt instructions** — invest in them.
- `JsonOutputParser` is a lighter alternative when you want raw dict/list output.

**Next:** [05 — LCEL & Runnables](05_LCEL_and_Runnables.ipynb). We've been quietly using
`|` to connect components; now we'll understand the composition system that powers it.